In [1]:
# Libraries and configurations
import sys
from pathlib import Path
import json

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision import transforms
from torchinfo import summary

# Visualization and analysis
import os
import seaborn as sns
import PIL
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import cohen_kappa_score

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [2]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()  # normalmente .../baseline_networks/notebooks
BASELINE_DIR = NOTEBOOK_DIR.parent   # .../baseline_networks
PROJECT_ROOT = BASELINE_DIR.parent   # .../flim_classification

sys.path.insert(0, str(PROJECT_ROOT))   # para importar config.py
sys.path.insert(0, str(BASELINE_DIR))   # para importar src.dataset

In [3]:
from config import get_dataset_paths, get_split_path_incremental
from src.dataset import DataModuleParasite
from src import utils
from src import models
from src import trainer

In [6]:
# Configuration for Eggs Dataset

CONFIG_EGG = {
    'dataset_name': 'eggs',
    'split': [1, 2, 3],
    'percentage': [1, 5, 25, 50, 75, 100],
    'num_classes': 9,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataset_name = CONFIG_EGG['dataset_name']
model_name = 'shufflenetv2_scratch'
type_model = 'shufflenetv2'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"
config = CONFIG_EGG
dataloaders = {}

transforms_egg = transforms.Compose([
    transforms.Resize((config['image_size'], config['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_egg)

model_scratch, _, _, _ = models.create_model(type_model=type_model, description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt", config=config, pre_trained=pre_trained)

n_flops, n_params = utils.count_flops(model_scratch, input_size=(1, 3, config['image_size'], config['image_size']))
print(f"Model: {model_name} - Dataset: {dataset_name}")
print(f"FLOPs: {n_flops} FLOPs")
print(f"Parameters: {n_params} ")

Model: shufflenetv2_scratch - Dataset: eggs
FLOPs: 133907224.0 FLOPs
Parameters: 1262829.0 


In [7]:
# Training loop with pre-trained weights
historic_train = trainer.train_loop(config, dataloaders, path, dataset_name, model_name=model_name, type_model=type_model, pre_trained=pre_trained)

# Save training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping at epoch 21 for split 1 and percentage 1%


Early stopping at epoch 22 for split 2 and percentage 1%


Early stopping at epoch 21 for split 3 and percentage 1%


Early stopping at epoch 28 for split 1 and percentage 5%


Early stopping at epoch 27 for split 2 and percentage 5%


...

Early stopping at epoch 48 for split 2 and percentage 75%


Early stopping at epoch 81 for split 3 and percentage 75%


Early stopping at epoch 58 for split 1 and percentage 100%


Early stopping at epoch 35 for split 2 and percentage 100%


Early stopping at epoch 57 for split 3 and percentage 100%


In [8]:
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

results_test = utils.evaluate_test_set(config, path, model_name, dataloaders, type_model=type_model, pre_trained=pre_trained)  

# Save results to JSON
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to shufflenetv2/shufflenetv2_scratch/eggs/shufflenetv2_scratch_aggregated_classification_report_eggs.txt
